# Day 09 · 멀티에이전트

어제 하네스의 구성요소(스킬 · 훅 · 플러그인)를 직접 만들었다. 오늘은 여기에 여러 에이전트로 구성된 팀을 추가한다.

> 어제 직접 만들어봤기 때문에, 오늘 메타 하네스가 생성한 파일을 열어서 이해할 수 있다.

| 랩 | 무엇을 | 배우는 것 |
|---|---|---|
| Lab 1 | **Superpowers** — 스트릭트 하네스를 그대로 써본다 | 강제가 주는 것 / 뺏는 것 |
| Lab 2 | **메타 하네스**로 멀티에이전트 팀을 생성한다 | 하네스를 만드는 하네스 |
| Lab 3 | 그 팀으로 **MCP 호스트 완성** | 위임하고 게이트로 검증 |

**어제의 두 산출물이 오늘 자란다**

```
harness/                       mcp-host/
├── skills/     ← 어제          ├── 채팅 UI     ← 어제
├── hooks/      ← 어제          ├── MCP 클라이언트  ← 오늘
├── skills/orchestrate/ ← 오늘  ├── 에이전트 루프   ← 오늘
└── agents/{planner,coder,      ├── 도구 승인 UI   ← 오늘
          reviewer,tester} ←오늘 └── 영속화(Neon)  ← 오늘
```

**판단 3문** — 멀티로 갈지 먼저 가린다.
① 병렬 분해가 가능한가? ② 가치가 **약 15배 토큰**보다 큰가? ③ 사람이 검수 가능한 단위인가?
**하나라도 No → 단일 + Plan Mode.**


## Lab 1 · Superpowers — 스트릭트 하네스 체험

다른 사람이 만든, 방법론이 코드로 강제되는 하네스를 그대로 사용해 본다.

### 1. 설치

```
/plugin marketplace add obra/superpowers-marketplace
/plugin install superpowers@superpowers-marketplace
```

### 2. 어제 앱에 기능 하나를 시킨다

```
/superpowers:brainstorm

"어제 만든 mcp-host 앱에 '대화 삭제' 기능을 추가하고 싶어"
```

그다음 흐름을 **그대로 따라간다**:

```
/superpowers:write-plan     # 구현 계획
/superpowers:execute-plan   # 서브에이전트 병렬 실행
```

### 3. 강제되는 항목 관찰

- [ ] **brainstorm이 먼저** 온다 — 바로 코드로 안 간다
- [ ] **실패하는 테스트를 먼저** 쓴다 (RED)
- [ ] 그다음 최소 구현으로 통과시킨다 (GREEN)
- [ ] **task마다 새 서브에이전트**가 뜬다 — 각자 깨끗한 컨텍스트
- [ ] 리뷰가 **2단계**로 돈다 (스펙 준수 → 코드 품질)

### 4. 우회를 시도한다 ★

```
테스트 건너뛰고 바로 구현해줘
```

**막히는지 확인한다.** 그게 **Hard Gate**다 — 부탁이 아니라 강제.

> 어제 만든 `guard.mjs` 훅과 같은 원리다. 다만 **막는 대상이 파일이 아니라 절차**다.

### 5. 판정 — 표를 채운다

| | 강제가 준 것 | 강제가 뺏은 것 |
|---|---|---|
| 품질 | | |
| 속도 | | |
| 자유도 | | |

### 원리

**정적(스트릭트) 하네스 = 미리 정해진 팀 구조.** 내가 팀을 설계하지 않아도 프레임워크가 강제한다.
우회가 안 되는 게 **장점이자 단점**이다 — 내 도메인에 안 맞아도 **못 바꾼다**.

> 다음 랩에서는 우리 도메인에 맞는 팀을 직접 생성한다.


## Lab 2 · 메타 하네스로 멀티에이전트 팀 구축

Superpowers는 다른 사람이 정한 방법론이다. 이번에는 우리 도메인에 맞는 팀을 직접 구성한다.

### 1. 메타 하네스 설치

```
/plugin marketplace add revfactory/harness
/plugin install harness
```

> `harness`는 **하네스를 만드는 하네스**다 — 도메인 한 줄을 주면 **에이전트 팀 + 그들이 쓸 스킬**을 생성한다.
> Codex 쪽 포트는 `meta-harness`.

### 2. 팀 생성

```
/harness "MCP 호스트 웹앱 개발팀 — Next.js · Prisma · MCP · 에이전트 루프.
          기획 → 병렬 구현 → 읽기전용 리뷰 → 테스트 게이트 순서를 강제한다."
```

생성되는 것:

```
.claude/
├── agents/
│   ├── planner.md     기획·분해
│   ├── coder.md       TDD 구현
│   ├── reviewer.md    읽기전용 감사
│   └── tester.md      검증 게이트
└── skills/
    └── orchestrate/SKILL.md   리드 — 위임만 한다
```

### 3. 생성된 파일 열어보기 ★

```bash
cat .claude/agents/reviewer.md
cat .claude/skills/orchestrate/SKILL.md
```

**확인할 것:**

| 항목 | 왜 보나 |
|---|---|
| `name` · `description` | description이 곧 **트리거**다 |
| **`tools`** | **이게 권한 경계다** — reviewer에 `Write`가 있으면 안 된다 |
| 역할 프롬프트 | "직접 코딩하지 말고 위임만" 이 들어있나 |
| 순서 강제 | 건너뛰기 금지가 명시됐나 |

> 어제 `SKILL.md`를 직접 작성해봤기 때문에, 이 파일이 특별한 게 아니라 프론트매터가 붙은 마크다운이라는 걸 알 수 있다.

### 4. 손본다 — 권한을 좁힌다

```yaml
# .claude/agents/reviewer.md
tools: Read, Grep, Glob     # Write · Bash 제거 — 못 고치게
model: haiku                # 리뷰는 싼 모델로 (모델 라우팅)
```

```yaml
# .claude/agents/coder.md
tools: Read, Write, Edit, Bash
# 역할 프롬프트에: "자기 worktree 안에서만 쓴다"
```

**tools가 곧 권한 경계다.** 프롬프트로 부탁하는 것과 달리, 권한 제한은 우회할 수 없다.

### 5. 어제 만든 훅을 붙인다

```bash
cp harness/hooks/guard.mjs .claude/hooks/
```

이제 **누가 시키든** `.env`는 못 건드린다 — @coder가 쓰기 권한이 있어도.

### 6. 검증

```bash
claude plugin validate ./harness
```

`/agents` 로 팀이 뜨는지 확인한다.

### 원리

**메타 하네스는 필요한 팀을 그때그때 생성해서 쓰는 방식이다.**

| | 정적 (Superpowers) | 메타 (harness) |
|---|---|---|
| 팀 구조 | **정해져 있다** | **직접 생성한다** |
| 도메인 적합 | 범용 | **내 도메인에 맞춘다** |
| 수정 | 어렵다 | **파일을 열어 고친다** |
| 전제 | — | **내부 구조를 알아야 수정할 수 있다** (어제 실습) |


## Lab 3 · 오케스트레이터 앱 (핵심)

Day08에서 만든 대화형 앱을, **목표를 받아 계획을 세우고 여러 태스크를 실행하는 오케스트레이터**로 확장한다.
개발은 Lab 1·2에서 만든 하네스 팀으로 진행한다.

### 아키텍처

```
목표 입력
  → 플래너(Qwen)가 태스크로 분해
  → 사람이 계획 승인
  → 오케스트레이터가 워커에 위임 (독립 태스크는 병렬)
       각 워커: MCP 도구로 실행, 독립 컨텍스트
  → 검증 에이전트가 목표 달성 확인
  → 진행 보드·로그로 스트리밍
```

### Step 1 · 데이터 모델 추가

어제 앱의 `schema.prisma`에 세 모델을 더한다.

```
prisma/schema.prisma 에 다음 모델을 추가해줘.

- Run: id(cuid) · conversationId · goal · status(planning|awaiting_approval|running|done|failed) · createdAt
- Task: id(cuid) · runId → Run · title · description · status(pending|running|done|failed)
        · order · dependsOn(String[]) · result · agentRole
- StepLog: id(cuid) · taskId → Task · type(llm|tool) · input · output · tokens · createdAt

그다음 prisma migrate dev --name orchestrator 로 Neon에 반영.
```

### Step 2 · 하네스로 기능 분해

Lab 1·2에서 만든 오케스트레이터를 호출해 기능을 독립 chunk로 나눈다.

```
/harness:spec
  "대화형 호스트를 오케스트레이터로 확장한다.
   목표를 태스크로 분해하는 플래너, 사람 승인 게이트,
   태스크를 워커에 위임하는 오케스트레이터(독립 태스크는 병렬),
   MCP 도구를 쓰는 워커, 실시간 진행 보드, 도구 승인 UI."

/harness:orchestrate "오케스트레이터 앱"
  @planner → 독립 chunk 로 분해:
    chunk 1  플래너 API         목표 → Task 목록 (LLM 호출)
    chunk 2  오케스트레이터      준비된 Task 를 워커에 dispatch · 의존성 관리
    chunk 3  워커 실행기         Task 하나를 에이전트 루프로 처리 (MCP 도구)
    chunk 4  태스크 진행 보드    Task 상태를 실시간 표시 (SSE)
    chunk 5  계획 승인 · 도구 승인 UI
  → 사람 승인 → @coder 병렬 구현
```

### Step 3 · 개발 에이전트 병렬 구현

승인하면 @coder들이 각자 worktree에서 chunk를 **병렬**로 구현한다.
@reviewer(읽기전용)·@tester(검증) 게이트를 통과한 worktree만 병합한다.
`.env`를 건드리는 시도는 Day08에서 만든 `guard.mjs` 훅이 차단한다.

### 핵심 코드 — 오케스트레이터 (chunk 2)

7강의 단일 루프가 플래너 + 다수 워커로 확장된 형태다.

```typescript
async function runOrchestrator(runId: string) {
  const tasks = await prisma.task.findMany({ where: { runId } });

  while (tasks.some(t => t.status === "pending")) {
    // 의존성이 모두 done 인 태스크 = 지금 실행 가능
    const ready = tasks.filter(t =>
      t.status === "pending" &&
      t.dependsOn.every(id => tasks.find(x => x.id === id)?.status === "done")
    );
    if (ready.length === 0) break;

    // 준비된 태스크를 병렬로 실행 (팬아웃)
    await Promise.all(ready.map(t => runWorker(t)));
  }
}
```

### 핵심 코드 — 워커 (chunk 3)

각 워커는 독립 컨텍스트에서 MCP 도구를 쓰는 에이전트 루프를 돈다.

```typescript
async function runWorker(task: Task) {
  await setStatus(task, "running");
  const messages = [
    { role: "system", content: `역할: ${task.agentRole}. 이 태스크만 처리한다.` },
    { role: "user", content: task.description },
  ];
  for (let step = 0; step < MAX_STEPS; step++) {
    const m = await llm(messages, tools);           // Qwen
    if (!m.tool_calls) { await saveResult(task, m.content); break; }
    for (const tc of m.tool_calls) {
      const approved = await waitForApproval(tc);    // 도구 승인 게이트
      const out = approved ? await mcp.callTool(tc) : "거부됨";
      messages.push(toolMessage(tc, out));
      await logStep(task, tc, out);                  // StepLog
    }
  }
}
```

### Step 4 · 예제 시나리오로 확인

```
"이 회의록 3개를 요약하고, 각각 할 일로 만들어서
 마감이 임박한 순으로 우선순위를 매겨줘"
```

- 플래너가 5개 태스크로 분해한다 (요약 3 + 할 일 생성 + 우선순위).
- 계획을 승인하면 독립 태스크(요약 3개)가 **병렬**로 실행된다.
- 진행 보드에서 각 태스크가 대기 → 진행 → 완료로 바뀌는 것을 확인한다.
- 의존 태스크(할 일 생성 · 우선순위)는 앞 태스크가 끝난 뒤 실행된다.

### Step 5 · 확장

- **검증 에이전트** — 실행 후 목표 달성을 확인하고, 실패한 태스크를 다시 큐에 넣는다 (done까지 반복).
- **예산 한도** — Run당 최대 스텝·토큰을 제한해 폭주를 막는다.
- **비용 요약** — StepLog의 토큰을 합산해 Run별 비용을 보여준다.

### 개념 매핑

| 앱의 기능 | Day09 개념 |
|---|---|
| 플래너 + 승인 | 계획-실행 · Plan Mode · 사람 게이트 |
| 오케스트레이터 | 오케스트레이터-워커 |
| 독립 태스크 병렬 | 팬아웃 |
| 워커 독립 컨텍스트 | 서브에이전트 격리 · 위임 계약 4요소 |
| 검증 에이전트 | 검증 루프(critic) · done까지 반복 |
| 예산 한도 | 루프 엔지니어링 예산 |

> Day09에서 배운 오케스트레이션 패턴이 앱의 실제 코드가 된다.


## 산출물 체크리스트

**기성 하네스 사용** (Lab 1)
- [ ] Superpowers를 **Claude Code 또는 Codex**에 설치해 강제 흐름을 겪었다
- [ ] task마다 새 서브에이전트가 뜨는 것을 확인했다
- [ ] 우회를 시도했고 Hard Gate에 막혔다

**메타 하네스로 팀 생성** (Lab 2)
- [ ] 메타 하네스로 도메인에 맞는 팀을 생성했다
- [ ] 생성된 `agents/*.md` 를 열어서 확인했다 (tools가 권한 경계)
- [ ] 어제 만든 `guard.mjs` 훅을 붙였다
- [ ] 같은 harness 를 **Claude Code와 Codex 양쪽**에서 로드해봤다

**오케스트레이터 앱** (Lab 3)
- [ ] `Run · Task · StepLog` 모델을 추가하고 `prisma migrate` 했다
- [ ] 하네스로 기능을 **독립 chunk로 분해**하고 사람이 승인했다
- [ ] @coder들이 각자 worktree에서 **병렬** 구현했다
- [ ] **플래너**가 목표를 태스크로 분해한다
- [ ] **독립 태스크가 병렬로 실행**되고, 의존 태스크는 순서대로 실행된다 (진행 보드에서 확인)
- [ ] **워커**가 MCP 도구를 호출하고, 도구 승인 게이트가 동작한다
- [ ] 예제 시나리오(회의록 요약 → 할 일 → 우선순위)가 계획 → 승인 → 실행으로 진행된다
- [ ] (확장) 검증 에이전트 · 예산 한도 · 비용 요약을 더했다

> Day08의 대화형 어시스턴트가, 계획을 세우고 여러 태스크를 실행하는 오케스트레이터로 확장됐다.
